# MLPKriging — Deep Kernel Learning (Octave)

**MLPKriging** combines a multi-layer perceptron (MLP) feature map with Gaussian-process
regression. The MLP takes *all* input dimensions jointly and maps them into a learned
`d_out`-dimensional feature space, after which a standard Kriging model is fitted.
This is also known as *Deep Kernel Learning*.

Steps:
1. Setup mlibkriging
2. Define the Branin function and plot it
3. Build a space-filling design and evaluate it
4. Fit an `MLPKriging` model
5. Predict on a fine grid and plot mean + uncertainty
6. Inspect model parameters

## 0. Setup mlibkriging

Make sure the `mLibKriging.mex` binary is built and on the Octave path.

In [1]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

mlibkriging loaded


## 1. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).

In [2]:
function z = branin(X)
  x1 = X(:,1) * 15 - 5;
  x2 = X(:,2) * 15;
  z = (x2 - 5/(4*pi^2) .* x1.^2 + 5/pi .* x1 - 6).^2 ...
      + 10 * (1 - 1/(8*pi)) .* cos(x1) + 10;
end

grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure;
contourf(G1, G2, z_true, 20);
colorbar;
title('True Branin function'); xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp3b0b1kzl/plots/tmpvp5xsoxv does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 2. Design of experiments

We sample $n = 30$ points using a random design.

In [3]:
rand('state', 42);
n = 30;
X = rand(n, 2);
y = branin(X);

figure;
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title(sprintf('%d design points on Branin', n));
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp3b0b1kzl/plots/tmp5r46sle1 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 3. Fit an MLPKriging model

We use `[16, 8]` (two hidden layers), `d_out=2` (2D feature space),
and `activation='selu'`.

In [4]:
mk = MLPKriging(y, X, [16, 8], 2, 'selu', 'matern5_2', 'constant', false, 'BFGS+Adam', 'LL');
disp(mk.summary())

* MLPKriging
  - kernel:      matern5_2
  - regmodel:    constant
  - normalize:   false
  - n obs:       30
  - d input:     2
  - d features:  2
  - warping:     MLPJoint(2 -> 16 -> 8 -> 2, 202 params)
  - sigma2:      2.67093e+10
  - theta:          9.1372   7.1890
  - beta:           8.4976e+04
  - LL:          -168.994
  - total warp params: 202



## 4. Predict on a fine grid

In [5]:
[p_mean, p_stdev] = mk.predict(grid_pts, true, false);
z_mean = reshape(p_mean, 50, 50);
z_sd   = reshape(p_stdev, 50, 50);

vmin = min(min(z_true(:)), min(z_mean(:)));
vmax = max(max(z_true(:)), max(z_mean(:)));

figure;
subplot(1, 2, 1);
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('True Branin'); xlabel('x_1'); ylabel('x_2');

subplot(1, 2, 2);
contourf(G1, G2, z_mean, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('MLPKriging mean'); xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp3b0b1kzl/plots/tmpngg7pi6a does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



In [6]:
% Posterior standard deviation (uncertainty)
figure;
contourf(G1, G2, z_sd, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title('MLPKriging std dev (uncertainty)');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp3b0b1kzl/plots/tmprd6zt4r1 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 5. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, log-likelihood,
hidden layer architecture, and activation function.

In [7]:
fprintf('Kernel       : %s\n', mk.kernel());
fprintf('Theta (range): %s\n', mat2str(mk.theta(), 4));
fprintf('Sigma2       : %.4f\n', mk.sigma2());
fprintf('LogLikelihood: %.4f\n', mk.logLikelihood());
fprintf('Feature dim  : %d\n', mk.feature_dim());
fprintf('Hidden dims  : %s\n', mat2str(mk.hidden_dims()));
fprintf('Activation   : %s\n', mk.activation());

Kernel       : matern5_2


Theta (range): [9.137;7.189]


Sigma2       : 26709318323.3646


LogLikelihood: -168.9937


Feature dim  : 2


Hidden dims  : [16 8]


Activation   : selu
